<a href="https://colab.research.google.com/github/anggajovansyah/learning/blob/main/UTS_INF1316_51240003%3C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluasi Tengah Semester INF-1316 Organisasi dan Arsitektur Komputer
Dosen: Mulya Agung
## Simulator Data Hazards CPU dan Virtual Memory

**Petunjuk**
1. Isi `NIM` pada sel berikut, lalu jalankan semua sel dari atas (*Run All*).
2. **Jangan mengubah** sel generator soal dan sel konstanta. Parameter soal berbeda untuk setiap NIM.
3. Lengkapi setiap fungsi bertanda `# TODO` pada soal A-D dan **hapus baris `raise NotImplementedError`**. Jangan mengganti nama atau parameter fungsi.
4. Kode akan diuji otomatis dengan program grader menggunakan NIM Anda **dan** lima NIM lain, jadi fungsi harus bekerja untuk parameter apa pun.
5. Kumpulkan notebook ini ke eCampuz dengan nama `UTS_INF1316_<NIM>.ipynb`.

In [6]:
NIM = "51240003"

## Generator soal (jangan diubah)

In [7]:
# JANGAN DIUBAH: generator parameter soal per NIM
import random

PAGE_SIZE = 4096
SKELETONS = [  # register abstrak a..g dipetakan ke R1..R7 secara acak per NIM
    [("I1", "LOAD",  "a", [],         "m0"),
     ("I2", "ADD",   "b", ["a", "c"], None),
     ("I3", "SUB",   "a", ["d", "e"], None),
     ("I4", "STORE", None, ["b"],     "m1"),
     ("I5", "ADD",   "b", ["f", "g"], None),
     ("I6", "LOAD",  "c", [],         "m2"),
     ("I7", "STORE", None, ["c"],     "m3")],
    [("I1", "LOAD",  "a", [],         "m0"),
     ("I2", "SUB",   "b", ["c", "d"], None),
     ("I3", "ADD",   "c", ["a", "b"], None),
     ("I4", "LOAD",  "a", [],         "m1"),
     ("I5", "STORE", None, ["c"],     "m2"),
     ("I6", "ADD",   "e", ["a", "f"], None),
     ("I7", "STORE", None, ["e"],     "m3")],
    [("I1", "ADD",   "a", ["b", "c"], None),
     ("I2", "LOAD",  "b", [],         "m0"),
     ("I3", "STORE", None, ["a"],     "m1"),
     ("I4", "SUB",   "a", ["b", "d"], None),
     ("I5", "LOAD",  "c", [],         "m2"),
     ("I6", "ADD",   "e", ["a", "c"], None),
     ("I7", "STORE", None, ["e"],     "m3")],
]

def _outcome(program, pt):
    """Hitung jumlah page fault & protection fault (mode user) untuk validasi."""
    faults = prot = 0
    for _, op, _, _, addr in program:
        if addr is None: continue
        pte = pt[addr // PAGE_SIZE]
        w = op == "STORE"
        if pte["sup"] or (w and not pte["write"]) or (not w and not pte["read"]):
            prot += 1
        elif not pte["present"]:
            faults += 1
    return faults, prot

def generate(nim):
    rng = random.Random(int("".join(ch for ch in str(nim) if ch.isdigit())))
    while True:
        skeleton = rng.choice(SKELETONS)
        regs = rng.sample([f"R{i}" for i in range(1, 8)], 7)
        rmap = dict(zip("abcdefg", regs))
        vpns = rng.sample(range(4), 4)              # VPN untuk m0..m3
        addr = {f"m{k}": vpns[k] * PAGE_SIZE + rng.randrange(0, PAGE_SIZE, 4) for k in range(4)}
        program = [(n, op, rmap.get(d), [rmap[s] for s in src], addr.get(m))
                   for n, op, d, src, m in skeleton]
        frames = rng.sample(range(1, 16), 6)
        absent = set(rng.sample(range(4), 2))
        pt = {}
        for v in range(4):
            pt[v] = {"present": v not in absent, "ppn": None if v in absent else frames[v],
                     "read": 1, "write": rng.randint(0, 1), "sup": int(rng.random() < 0.25)}
        free = frames[4:6]
        f, p = _outcome(program, pt)
        if f >= 1 and p >= 1:                        # pastikan ada page fault & protection fault
            return program, pt, free

In [8]:
PAGE_FAULT_PENALTY = 100  # cycle

class PageFault(Exception): pass
class ProtectionFault(Exception): pass

PROGRAM, PAGE_TABLE, FREE_FRAMES = generate(NIM)

## Parameter soal Anda

In [9]:
print("Program:")
for name, op, dest, srcs, addr in PROGRAM:
    print(f"  {name}: {op:<5} tujuan={dest or '-':<4} sumber={', '.join(srcs) or '-':<8} alamat={hex(addr) if addr is not None else '-'}")
print("\nPage table (page size 4 KB, mode user):")
for vpn, e in PAGE_TABLE.items():
    print(f"  VPN {vpn}: present={int(e['present'])} ppn={e['ppn']} READ={e['read']} WRITE={e['write']} SUP={e['sup']}")
print("\nFREE_FRAMES =", FREE_FRAMES)

Program:
  I1: LOAD  tujuan=R4   sumber=-        alamat=0xf68
  I2: ADD   tujuan=R1   sumber=R4, R5   alamat=-
  I3: SUB   tujuan=R4   sumber=R6, R3   alamat=-
  I4: STORE tujuan=-    sumber=R1       alamat=0x1650
  I5: ADD   tujuan=R1   sumber=R2, R7   alamat=-
  I6: LOAD  tujuan=R5   sumber=-        alamat=0x36f8
  I7: STORE tujuan=-    sumber=R5       alamat=0x26e8

Page table (page size 4 KB, mode user):
  VPN 0: present=0 ppn=None READ=1 WRITE=1 SUP=0
  VPN 1: present=1 ppn=13 READ=1 WRITE=0 SUP=0
  VPN 2: present=1 ppn=11 READ=1 WRITE=1 SUP=0
  VPN 3: present=0 ppn=None READ=1 WRITE=1 SUP=0

FREE_FRAMES = [5, 4]


## A. Deteksi data hazards (24 poin)
Kembalikan list tuple `(jenis, instr_awal, instr_akhir, register)` untuk **semua** pasangan (i, j) dengan i < j:
- **RAW**: register tujuan Ii dibaca Ij
- **WAR**: register sumber Ii ditulis Ij
- **WAW**: Ii dan Ij menulis register yang sama

In [17]:
def detect_hazards(program):
  n = len(program)
  hazards=[]

  for i in range(n):
      for j in range(i + 1, n):
          name_i, _, dest_i, srcs_i, _ = program[i]
          name_j, _, dest_j, srcs_j, _ = program[j]

          # 1. Cek RAW
          if dest_i is not None and dest_i in srcs_j:
            hazards.append(("RAW", name_i, name_j, dest_i))

          # 2. Cek WAR
          if dest_j is not None and dest_j in srcs_i:
              hazards.append(("WAR", name_i, name_j, dest_j))

          # 3. Cek WAW
          if dest_i is not None and dest_i == dest_j:
              hazards.append(("WAW", name_i, name_j, dest_i))

  return hazards

hasil_hazard = detect_hazards(PROGRAM)
for h in hasil_hazard:
    print(h)

('RAW', 'I1', 'I2', 'R4')
('WAW', 'I1', 'I3', 'R4')
('WAR', 'I2', 'I3', 'R4')
('RAW', 'I2', 'I4', 'R1')
('WAW', 'I2', 'I5', 'R1')
('WAR', 'I2', 'I6', 'R5')
('WAR', 'I4', 'I5', 'R1')
('RAW', 'I6', 'I7', 'R5')


## B. Register renaming (18 poin)
Ganti setiap register tujuan dengan register fisik baru `P0, P1, ...` sesuai urutan, dan perbarui register sumber pada instruksi berikutnya mengikuti pemetaan terbaru.

In [18]:
def rename_registers(program):
    mapping = {}
    counter = 0
    renamed_program = []

    for name, op, dest, srcs, addr in program:
        # 1. Perbarui register sumber mengikuti pemetaan terbaru
        new_srcs = [mapping.get(s, s) for s in srcs]

        # 2. Ganti register tujuan dengan register fisik baru (P0, P1, ...)
        if dest is not None:
            new_dest = f"P{counter}"
            mapping[dest] = new_dest
            counter += 1
        else:
            new_dest = None

        # 3. Simpan tuple instruksi yang telah diperbarui
        renamed_program.append((name, op, new_dest, new_srcs, addr))

    return renamed_program


renamed = rename_registers(PROGRAM)
for ins in renamed:
    print(ins)
print("Hazard tersisa:", detect_hazards(renamed))

('I1', 'LOAD', 'P0', [], 3944)
('I2', 'ADD', 'P1', ['P0', 'R5'], None)
('I3', 'SUB', 'P2', ['R6', 'R3'], None)
('I4', 'STORE', None, ['P1'], 5712)
('I5', 'ADD', 'P3', ['R2', 'R7'], None)
('I6', 'LOAD', 'P4', [], 14072)
('I7', 'STORE', None, ['P4'], 9960)
Hazard tersisa: [('RAW', 'I1', 'I2', 'P0'), ('RAW', 'I2', 'I4', 'P1'), ('RAW', 'I6', 'I7', 'P4')]


## C. Virtual memory (34 poin)
**`translate`** (24 poin): hitung VPN = `vaddr // PAGE_SIZE` dan offset. Periksa berurutan:
1. `SUP = 1` dan bukan supervisor → `ProtectionFault`
2. akses `"read"` dengan `READ = 0`, atau `"write"` dengan `WRITE = 0` → `ProtectionFault`
3. `present = False` → `PageFault(vpn)`

Kembalikan alamat fisik = `PPN * PAGE_SIZE + offset`.

**`handle_page_fault`** (10 poin): ambil frame kosong pertama dari `free_frames`, tulis sebagai PPN, set `present = True`.

In [19]:
def translate(vaddr, access, page_table, supervisor=False):
    vpn = vaddr // PAGE_SIZE
    offset = vaddr % PAGE_SIZE
    pte = page_table[vpn]

    # 1. Cek hak akses supervisor
    if pte["sup"] and not supervisor:
        raise ProtectionFault("Akses supervisor ditolak pada mode user")

    # 2. Cek izin baca/tulis
    if (access == "read" and not pte["read"]) or (access == "write" and not pte["write"]):
        raise ProtectionFault(f"Izin {access} ditolak pada VPN {vpn}")

    # 3. Cek ketersediaan halaman di memori fisik
    if not pte["present"]:
        raise PageFault(vpn)

    return pte["ppn"] * PAGE_SIZE + offset


def handle_page_fault(vpn, page_table, free_frames):
    ppn = free_frames.pop(0)
    page_table[vpn]["ppn"] = ppn
    page_table[vpn]["present"] = True
    return ppn


for name, op, _, _, addr in PROGRAM:
    if addr is None:
        continue
    access = "read" if op == "LOAD" else "write"
    try:
        print(name, hex(addr), "->", hex(translate(addr, access, PAGE_TABLE)))
    except PageFault as e:
        print(name, hex(addr), "-> PageFault pada VPN", e.args[0])
    except ProtectionFault as e:
        print(name, hex(addr), "-> ProtectionFault:", e)

I1 0xf68 -> PageFault pada VPN 0
I4 0x1650 -> ProtectionFault: Izin write ditolak pada VPN 1
I6 0x36f8 -> PageFault pada VPN 3
I7 0x26e8 -> 0xb6e8


## D. Eksekusi dan kinerja (24 poin)
**`raw_stalls`** (8 poin): stall setiap pasangan RAW = `max(0, 3 - jarak)`, dengan jarak = selisih indeks instruksi.

**`run`** (16 poin): jalankan program, kembalikan `(total_cycles, jumlah_page_fault, total_stall, log)`.
- 1 cycle per instruksi (termasuk yang dibatalkan), +100 cycle per page fault, ditambah total stall RAW.
- Setelah page fault: tangani, lalu **ulangi** instruksi.
- Protection fault: catat di log, instruksi dibatalkan, lanjut ke instruksi berikutnya.
- **Salin** `page_table` dan `free_frames` sebelum diubah; jangan mengubah milik pemanggil.

In [20]:
import copy

def raw_stalls(program):
    # Petakan nama instruksi ("I1", "I2", ...) ke indeksnya (0, 1, ...)
    idx_map = {ins[0]: idx for idx, ins in enumerate(program)}
    total_stall = 0

    for jenis, instr_awal, instr_akhir, _ in detect_hazards(program):
        if jenis == "RAW":
            jarak = idx_map[instr_akhir] - idx_map[instr_awal]
            total_stall += max(0, 3 - jarak)

    return total_stall


def run(program, page_table, free_frames, supervisor=False):
    # Salin page_table dan free_frames agar tidak mengubah milik pemanggil
    pt = copy.deepcopy(page_table)
    frames = list(free_frames)

    total_cycles = 0
    jumlah_page_fault = 0
    total_stall = raw_stalls(program)
    log = []

    for name, op, dest, srcs, addr in program:
        # 1 cycle per instruksi (termasuk yang dibatalkan)
        total_cycles += 1

        # Jika instruksi tidak mengakses memori (ADD / SUB)
        if addr is None:
            log.append(f"{name}: {op} selesai")
            continue

        access = "read" if op == "LOAD" else "write"

        # Gunakan loop agar jika terjadi PageFault, instruksi diulangi setelah ditangani
        while True:
            try:
                paddr = translate(addr, access, pt, supervisor=supervisor)
                log.append(f"{name}: {op} {hex(addr)} -> {hex(paddr)}")
                break
            except PageFault as e:
                vpn = e.args[0]
                jumlah_page_fault += 1
                total_cycles += PAGE_FAULT_PENALTY
                ppn = handle_page_fault(vpn, pt, frames)
                log.append(f"{name}: PageFault pada VPN {vpn}, dialokasikan ke PPN {ppn}")
            except ProtectionFault as e:
                log.append(f"{name}: ProtectionFault ({e}), instruksi dibatalkan")
                break

    total_cycles += total_stall
    return total_cycles, jumlah_page_fault, total_stall, log


cycles, faults, stalls, log = run(PROGRAM, PAGE_TABLE, FREE_FRAMES)
for entry in log:
    print(entry)
print(f"\npage fault = {faults}, stall RAW = {stalls}, total cycle = {cycles}")

I1: PageFault pada VPN 0, dialokasikan ke PPN 5
I1: LOAD 0xf68 -> 0x5f68
I2: ADD selesai
I3: SUB selesai
I4: ProtectionFault (Izin write ditolak pada VPN 1), instruksi dibatalkan
I5: ADD selesai
I6: PageFault pada VPN 3, dialokasikan ke PPN 4
I6: LOAD 0x36f8 -> 0x46f8
I7: STORE 0x26e8 -> 0xb6e8

page fault = 2, stall RAW = 5, total cycle = 212
